# Abstract Fcatory Pattern

### THE ABSTRACT PRODUCTS (Interfaces)

These interfaces define the "Contract". Any new vehicle or navigation system </br>
added later must adhere to these rules. The client code only talks to these. </br> 

In [1]:
from abc import ABC, abstractmethod

class Vehicle(ABC):
    """
    Abstract Product A: Defines the interface for all vehicle types.
    """
    @abstractmethod
    def move_cargo(self) -> str:
        pass

class Navigation(ABC):
    """
    Abstract Product B: Defines the interface for all navigation controls.
    """
    @abstractmethod
    def calculate_route(self) -> str:
        pass

#### THE CONCRETE PRODUCT FAMILIES

These are the actual implementations. Note that we have two distinct families:
- The Road Family (Truck + GPS)
- The Sea Family (Ship + Sonar)

In [2]:
# --- Family 1: Road Implementation ---
class Truck(Vehicle):
    def move_cargo(self) -> str:
        return "Truck: Driving on highway with 18 wheels."

class GPS(Navigation):
    def calculate_route(self) -> str:
        return "GPS: Calculating overload road path via satellites."

# --- Family 2: Sea Implementation ---
class Ship(Vehicle):
    def move_cargo(self) -> str:
        return "Ship: Sailing across the Atlantic ocean."

class Sonar(Navigation):
    def calculate_route(self) -> str:
        return "Sonar: Checking water depth and coral reefs."

### THE ABSTRACT FACTORY (The Blueprint)

In [3]:
class TransportFactory(ABC):
    """
    The Abstract Factory Interface.
    
    This is the core of the pattern. It declares a set of methods that return
    different abstract products. 
    
    Crucial Point: It forces any concrete factory to create *both* a Vehicle 
    and a Navigation system, ensuring they always come in pairs.
    """
    @abstractmethod
    def create_vehicle(self) -> Vehicle:
        pass

    @abstractmethod
    def create_navigation(self) -> Navigation:
        pass

### THE CONCRETE FACTORIES (The Manufacturers)

These classes implement the creation logic. Each factory is responsible for
creating one specific variant of the product family.

In [4]:
class RoadFactory(TransportFactory):
    """
    Concrete Factory 1: Produces only Road-compatible objects.
    Guarantees that if you get a Vehicle from here, it's a Truck.
    """
    def create_vehicle(self) -> Vehicle:
        return Truck()

    def create_navigation(self) -> Navigation:
        return GPS()

class SeaFactory(TransportFactory):
    """
    Concrete Factory 2: Produces only Sea-compatible objects.
    Guarantees that if you get a Vehicle from here, it's a Ship.
    """
    def create_vehicle(self) -> Vehicle:
        return Ship()

    def create_navigation(self) -> Navigation:
        return Sonar()

### CLIENT CODE (The Application)

In [5]:
class LogisticsApplication:
    """
    The Client. 
    
    NOTICE: This class knows NOTHING about 'Truck', 'Ship', 'RoadFactory', etc.
    It only knows about the abstract types 'TransportFactory', 'Vehicle', and 
    'Navigation'.
    
    This is 'Dependency Injection' - we inject the factory we want to use.
    """
    def __init__(self, factory: TransportFactory):
        self.factory = factory

    def start_mission(self) -> None:
        # 1. Create the objects using the injected factory
        vehicle: Vehicle = self.factory.create_vehicle()
        nav: Navigation = self.factory.create_navigation()

        # 2. Use the objects (The logic remains the same regardless of factory type)
        print(f"Vehicle Status: {vehicle.move_cargo()}")
        print(f"Nav Status:     {nav.calculate_route()}")
        print("-" * 50)

### EXECUTION

In [6]:
def main():
    # Imagine this setting is read from a 'config.json' file or environment variable
    # Options: "road" or "sea"
    current_os_config = "sea" 

    print(f"--- Booting Logistics App (Mode: {current_os_config.upper()}) ---\n")

    # The Factory Selection Logic
    # This is usually the ONLY place in your code that knows about concrete classes.
    selected_factory: TransportFactory

    match current_os_config:
        case "road":
            selected_factory = RoadFactory()
        case "sea":
            selected_factory = SeaFactory()
        case _:
            raise ValueError("Unknown configuration type")

    # Pass the factory to the application.
    # The app works automatically with whatever factory we gave it.
    app = LogisticsApplication(selected_factory)
    app.start_mission()

if __name__ == "__main__":
    main()

--- Booting Logistics App (Mode: SEA) ---

Vehicle Status: Ship: Sailing across the Atlantic ocean.
Nav Status:     Sonar: Checking water depth and coral reefs.
--------------------------------------------------


# More Pythonic Way

Yes. The previous examples followed the `"Classic" (Gang of Four)` implementation. While correct, it is often criticized in Python for being too verbose and "Java-like" (too many classes, heavy inheritance).

The Pythonic way leverages the fact that Classes and Functions are first-class citizens. You don't need a Factory Class to create objects; you can pass the Class types themselves or use simple functions.

Here is the Modern Pythonic Approach using Protocols (structural typing) and Functional Factories.

The Pythonic Changes
- `Protocol` instead of `ABC`: We don't force classes to inherit from specific parents. As long as they have the right methods (Duck Typing), they work.
- `dict` instead of `Factory Classes`: We map the logic in a simple dictionary.
- Classes as Data: We pass the class Truck directly, rather than making a method `create_truck()`.


#### Why `frozen=True`?
In the `@dataclass(frozen=True)` decorator, the frozen parameter makes the object immutable (unchangeable) after it is created.

#### We use it here for three key reasons:
- Safety (Configuration Integrity): A Factory definition is like a rulebook. `"Road"` implies Truck and GPS. You don't want a buggy part of your code to accidentally overwrite `family.vehicle_class = Ship` later in the execution. If you try to change a frozen object, Python raises a `FrozenInstanceError`.
- Intent: It signals to other developers: "This object is just a container for constant data. Read it, but don't touch it."
- Hashability: Because it is immutable, a frozen dataclass can be used as a key in a dictionary or added to a set (though in this specific example, we use it as a value).

### PROTOCOLS (The "Duck Typing" Interface)

We use Protocols instead of Abstract Base Classes. </br>
This means classes don't need to inherit from anything; </br>
they just need to have the matching methods.</br>

In [11]:
from typing import Protocol, runtime_checkable

@runtime_checkable
class Vehicle(Protocol):
    def move_cargo(self) -> str: ...

@runtime_checkable
class Navigation(Protocol):
    def calculate_route(self) -> str: ...


### CONCRETE CLASSES (The Implementation)

These are standalone classes. They are NOT instantiated here.

In [12]:
class Truck:
    def move_cargo(self) -> str: 
        return "Truck driving on highway"

class GPS:
    def calculate_route(self) -> str: 
        return "GPS calculating path via satellite"

class Ship:
    def move_cargo(self) -> str: 
        return "Ship sailing on waves"

class Sonar:
    def calculate_route(self) -> str: 
        return "Sonar checking water depth"

### THE PYTHONIC FACTORY (Registry)

In [13]:
@dataclass(frozen=True)
class TransportFamily:
    """
    A strictly typed container that holds the class references.
    frozen=True prevents accidental changes to the factory rules.
    """
    vehicle_class: Type[Vehicle]  # Stores the Class itself, not an instance
    nav_class: Type[Navigation]

# The Registry: Maps a string key to a family of classes.
# This replaces the need for "RoadFactory" and "SeaFactory" classes.
FACTORIES = {
    "road": TransportFamily(vehicle_class=Truck, nav_class=GPS),
    "sea":  TransportFamily(vehicle_class=Ship,  nav_class=Sonar),
}

def get_factory(mode: str) -> TransportFamily:
    """Retrieves the correct family based on configuration string."""
    try:
        return FACTORIES[mode]
    except KeyError:
        raise ValueError(f"Unknown mode: {mode}")

### CLIENT CODE

In [14]:
from typing import Protocol, Type, runtime_checkable
from dataclasses import dataclass


def run_mission(family: TransportFamily):
    """
    The client code receives the factory family.
    It performs the instantiation (adding parentheses) here.
    """
    print(f"--- Initializing Mission ---")
    
    # INSTANTIATION HAPPENS HERE
    vehicle_instance = family.vehicle_class() 
    nav_instance = family.nav_class()

    # USAGE
    print(f"Vehicle: {vehicle_instance.move_cargo()}")
    print(f"Nav:     {nav_instance.calculate_route()}")
    print("-" * 30)

def main():
    # Scenario 1: Road
    print("User configures: ROAD")
    road_tools = get_factory("road")
    run_mission(road_tools)

    # Scenario 2: Sea
    print("User configures: SEA")
    sea_tools = get_factory("sea")
    run_mission(sea_tools)

if __name__ == "__main__":
    main()

User configures: ROAD
--- Initializing Mission ---
Vehicle: Truck driving on highway
Nav:     GPS calculating path via satellite
------------------------------
User configures: SEA
--- Initializing Mission ---
Vehicle: Ship sailing on waves
Nav:     Sonar checking water depth
------------------------------
